# Dorado: Two-Stage Reasoning Post-Training

In [1]:
# ── Setup: environment, deps, GPU ────────────────────────────────────
import os, sys, shutil, subprocess
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Runtime cache
repo_candidates = ["/home/jovyan/llmpt", os.getcwd()]
repo_root = next((p for p in repo_candidates if os.path.isdir(p)), os.getcwd())
cache_dir = os.path.join(repo_root, ".runtime_cache")
Path(cache_dir).mkdir(parents=True, exist_ok=True)

for key, sub in [("DORADO_RUNTIME_CACHE", ""), ("HF_HOME", "huggingface"),
                  ("HF_HUB_CACHE", "huggingface/hub"), ("HF_DATASETS_CACHE", "huggingface/datasets"),
                  ("TRANSFORMERS_CACHE", "huggingface/transformers"), ("PIP_CACHE_DIR", "pip")]:
    os.environ[key] = os.path.join(cache_dir, sub) if sub else cache_dir

# Install deps via subprocess (no kernel restart needed)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.46.0,<5.0.0", "trl>=0.11.0,<0.14.0",
    "accelerate", "datasets", "peft", "bitsandbytes", "huggingface_hub",
    "sympy==1.12", "latex2sympy2", "word2number", "regex", "openpyxl",
    "filelock>=3.12.0"])

# GPU selection
try:
    rows = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,memory.free", "--format=csv,noheader,nounits"],
        text=True).strip().splitlines()
    gpus = sorted([(r.split(",")[0].strip(), int(r.split(",")[1])) for r in rows],
                   key=lambda x: x[1], reverse=True)
    selected = [g[0] for g in gpus[:8]]
except Exception:
    selected = ["0"]
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(selected)

# Ensure dorado is importable
for p in [os.path.abspath("."), os.path.expanduser("~/llmpt")]:
    if os.path.isdir(os.path.join(p, "dorado")):
        if p not in sys.path: sys.path.insert(0, p)
        os.chdir(p)
        break

import torch
print(f"🧩 GPU={os.environ['CUDA_VISIBLE_DEVICES']}  torch={torch.__version__}")
if torch.cuda.is_available():
    print(f"🎯 {torch.cuda.get_device_name(0)} ({torch.cuda.device_count()} visible)")

from dorado import (get_profile, PROFILES, build_experiment_grid, make_results_paths,
    run_sft_stage, run_candidate_generation, run_labeling_stage,
    run_dpo_training, run_full_evaluation,
    run_single_experiment, run_all_experiments, cleanup_artifacts,
    clear_gpu, set_random_seeds, cleanup_storage)
print(f"✅ dorado loaded (profiles: {list(PROFILES.keys())})")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.20.1+cu121 requires torch==2.5.1, but you have torch 2.4.1 which is incompatible.
torchaudio 2.5.1+cu121 requires torch==2.5.1, but you have torch 2.4.1 which is incompatible.


🧩 GPU=0  torch=2.4.1+cu121
🎯 NVIDIA GeForce RTX 3090 (1 visible)
✅ dorado loaded (profiles: ['smoke', 'fast', 'full'])


In [2]:
# ── Configure ────────────────────────────────────────────────────────
PROFILE = "smoke"  # "smoke" | "fast" | "full"
OVERRIDES = None    # e.g. {"math_prompt_count": 100}

EXPERIMENTS = build_experiment_grid(profile=PROFILE, overrides=OVERRIDES)
_, RESULTS_FILE, CHECKPOINT_FILE = make_results_paths()

cfg = EXPERIMENTS[0]
print(f"Profile: {PROFILE}  |  Model: {cfg['base_model']}  |  RM: {cfg['rm_strategy']}")
print(f"SFT: {cfg['sft_samples']} samples  |  Math: {cfg['math_prompt_count']} prompts")
print(f"DPO: β={cfg['dpo_beta']} lr={cfg['dpo_lr']}  |  Eval: {cfg['eval_benchmarks']}")
print(f"Results → {RESULTS_FILE}")

Profile: smoke  |  Model: HuggingFaceTB/SmolLM2-135M  |  RM: skip
SFT: 50 samples  |  Math: 20 prompts
DPO: β=0.1 lr=5e-07  |  Eval: ['math']
Results → results/2026-03-25/dorado_results.xlsx


In [3]:
# ── Run ───────────────────────────────────────────────────────────────
results_df = run_all_experiments(EXPERIMENTS, RESULTS_FILE, CHECKPOINT_FILE)

/opt/conda/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


ModuleNotFoundError: Could not import module 'Trainer'. Are this object's requirements defined correctly?

In [ ]:
# ── Results ───────────────────────────────────────────────────────────
if results_df is not None and not results_df.empty:
    cols = [c for c in results_df.columns if 'accuracy' in c or c in ['experiment_id', 'status', 'runtime_minutes', 'improvement_over_base']]
    display(results_df[cols])
else:
    print("No results yet.")
cleanup_artifacts()